# 7-3b 결과 확인
- 전체 감성 분포
- 토픽 배정 vs 아웃라이어 분리
- 토픽별 감성 분포

In [12]:
from google.colab import drive

import pandas as pd
import numpy as np

In [13]:
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/sparta/tp4/review_analysis/data/'

print(f'DATA_DIR: {DATA_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR: /content/drive/MyDrive/sparta/tp4/review_analysis/data/


## 1. 결과 로드

In [14]:
df = pd.read_parquet(DATA_DIR + 'step7_3b_review_sentiment.parquet')

print(f'전체 건수: {len(df):,}건')
print(f'컬럼: {df.columns.tolist()}')
print(f'\n샘플:')
print(df.head(3).to_string())

전체 건수: 623,380건
컬럼: ['리뷰번호', '리뷰내용_clean', 'goodsNo', '리뷰타입', '평점', '작성일', '사이즈', '화면대비색감', '퀄리티', '구김', '두께감', '신축성', '색감', '보온성', '퀄리티_점수', '보온성_점수', '신축성_점수', '두께감_점수', '구김_점수', '사이즈_편차절대', '화면대비색감_편차절대', '색감_편차절대', '만족도_응답여부', 'laugh_count', 'cry_count', 'exclamation_count', 'question_count', 'emoji_count', '성별', '구매사이즈', '구매상세', '사진유무', '도움돼요', '체험단', 'p_pos', 'p_neg', 'p_neu', 'sentiment']

샘플:
       리뷰번호                                            리뷰내용_clean  goodsNo   리뷰타입   평점                  작성일   사이즈 화면대비색감   퀄리티    구김   두께감   신축성    색감   보온성  퀄리티_점수  보온성_점수  신축성_점수  두께감_점수  구김_점수  사이즈_편차절대  화면대비색감_편차절대  색감_편차절대 만족도_응답여부  laugh_count  cry_count  exclamation_count  question_count  emoji_count       성별 구매사이즈   구매상세   사진유무  도움돼요    체험단     p_pos     p_neg     p_neu sentiment
0  34612047                        요즘 입기 좋은 것 같아요 무난무난하게 잘 입고있습니다  1733275  goods  5.0  2022-11-07 18:13:53  None   None  None  None  None  None  None  None     NaN     NaN     NaN     NaN    NaN       NaN

## 2. 전체 감성 분포

In [15]:
print('=== 전체 감성 분포 ===')
dist = df['sentiment'].value_counts()
for label, cnt in dist.items():
    print(f'  {label:10s}: {cnt:>7,}건 ({cnt/len(df)*100:.1f}%)')

=== 전체 감성 분포 ===
  positive  : 502,332건 (80.6%)
  negative  :  63,631건 (10.2%)
  neutral   :  57,417건 (9.2%)


In [16]:
# review_ids 로드
review_ids = np.load(DATA_DIR + 'step3_2_review_ids.npy')
print(f'review_ids 길이: {len(review_ids):,}')

# 매핑 테이블 생성
label_df = pd.DataFrame({
    '리뷰번호': review_ids,
    'topic_id': cluster_labels
})
label_df['리뷰번호'] = label_df['리뷰번호'].astype(str)
df['리뷰번호'] = df['리뷰번호'].astype(str)

# merge
df = df.merge(label_df, on='리뷰번호', how='left')
df['topic_id'] = df['topic_id'].fillna(-1).astype(int)

print(f'topic_id 붙이기 완료')
print(f'토픽 배정: {(df["topic_id"] != -1).sum():,}건')
print(f'아웃라이어: {(df["topic_id"] == -1).sum():,}건')

review_ids 길이: 598,301
topic_id 붙이기 완료
토픽 배정: 409,986건
아웃라이어: 213,394건


## 3. 토픽 배정 vs 아웃라이어 분리

In [17]:
# # cluster_labels 로드해서 붙이기
# import numpy as np
# cluster_labels = np.load(DATA_DIR + 'step5_1_cluster_labels.npy')
# df['topic_id'] = cluster_labels

# topic_df   = df[df['topic_id'] != -1].copy()
# outlier_df = df[df['topic_id'] == -1].copy()

# print(f'토픽 배정: {len(topic_df):,}건 ({len(topic_df)/len(df)*100:.1f}%)')
# print(f'아웃라이어: {len(outlier_df):,}건 ({len(outlier_df)/len(df)*100:.1f}%)')

# print('\n=== 토픽 배정 감성 분포 ===')
# dist = topic_df['sentiment'].value_counts()
# for label, cnt in dist.items():
#     print(f'  {label:10s}: {cnt:>7,}건 ({cnt/len(topic_df)*100:.1f}%)')

# print('\n=== 아웃라이어 감성 분포 ===')
# dist = outlier_df['sentiment'].value_counts()
# for label, cnt in dist.items():
#     print(f'  {label:10s}: {cnt:>7,}건 ({cnt/len(outlier_df)*100:.1f}%)')

## 4. 토픽별 감성 분포

In [19]:
topic_df   = df[df['topic_id'] != -1].copy()
outlier_df = df[df['topic_id'] == -1].copy()

In [20]:
# 토픽 키워드 로드 (토픽 이름 생성)
kw_df = pd.read_csv(DATA_DIR + 'step6_1_topic_keywords.csv')
topic_names = (
    kw_df.groupby('topic_id')
    .apply(lambda g: ' / '.join(
        g[g['category'] == 'core'].nsmallest(3, 'rank')['word'].tolist()
        or g.nsmallest(3, 'rank')['word'].tolist()
    ))
    .reset_index()
    .rename(columns={0: 'topic_name', 'topic_id': 'topic_id'})
)

# 토픽별 감성 분포 계산
topic_sent = (
    topic_df.groupby(['topic_id', 'sentiment'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
topic_sent['total'] = topic_sent.drop(columns='topic_id').sum(axis=1)

# 비율 계산
sent_cols = [c for c in topic_sent.columns if c not in ['topic_id', 'total']]
for s in sent_cols:
    topic_sent[f'{s}_pct'] = (topic_sent[s] / topic_sent['total'] * 100).round(1)

# 토픽 이름 붙이기
topic_sent = topic_sent.merge(topic_names, on='topic_id', how='left')
topic_sent = topic_sent.sort_values('total', ascending=False).reset_index(drop=True)

print('=== 토픽별 감성 분포 (리뷰 수 상위 20개) ===')
disp_cols = ['topic_id', 'topic_name', 'total'] + [f'{s}_pct' for s in sent_cols]
print(topic_sent[disp_cols].head(20).to_string(index=False))

=== 토픽별 감성 분포 (리뷰 수 상위 20개) ===
 topic_id              topic_name  total  negative_pct  neutral_pct  positive_pct
        6          바지 / 팬츠 / 바지 핏  30248           9.1          8.3          82.6
        2         배송 빠르 / 빠르 / 배송  28411          10.2          2.7          87.1
       15         후드티 / 후드 / 후드집업  19246           5.5          6.7          87.7
       64        워싱 / 화면 / 워싱 들어가  18853           2.4          3.5          94.1
       13      기모 따뜻 / 기모 / 기모 겨울  18682           9.4          7.8          82.8
       46               l / m / s  15609          13.7         23.7          62.7
       63       세탁 / 건조기 / 건조기 돌리  14480          43.0         13.9          43.1
       19         블랙 / 아이보리 / 그레이  14385           9.9          9.2          80.9
       51     가격 퀄리티 / 핏 가격 / 퀄리티  13723           2.6          3.8          93.6
       12      따뜻 핏 / 사이즈 따뜻 / 보온  12355           3.2          3.8          93.1
       61        작게 / 치수 / 사이즈 작게   9607          18.9         32.

/tmp/ipykernel_4959/651161213.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: ' / '.join(


## 5. 부정 비율 높은 토픽 TOP 10

In [21]:
# 300건 이상 토픽만
filtered = topic_sent[topic_sent['total'] >= 300].copy()

if 'negative_pct' in filtered.columns:
    print('=== 부정 비율 TOP 10 ===')
    top_neg = filtered.nlargest(10, 'negative_pct')
    for _, r in top_neg.iterrows():
        print(f"  T{int(r['topic_id']):2d} [{int(r['total']):>5,}건] "
              f"부정 {r['negative_pct']:5.1f}% "
              f"긍정 {r.get('positive_pct', 0):5.1f}% "
              f"중립 {r.get('neutral_pct', 0):5.1f}% "
              f"| {r['topic_name']}")

=== 부정 비율 TOP 10 ===
  T63 [14,480건] 부정  43.0% 긍정  43.1% 중립  13.9% | 세탁 / 건조기 / 건조기 돌리
  T38 [6,947건] 부정  33.0% 긍정  52.3% 중립  14.6% | 늘어나 / 목폴라 / 머리
  T58 [3,313건] 부정  28.5% 긍정  54.2% 중립  17.4% | 기장 / 밑위 / 기장 편하
  T29 [6,796건] 부정  28.1% 긍정  48.2% 중립  23.7% | 여름 힘들 / 두께감 여름 / 여름
  T55 [4,368건] 부정  27.6% 긍정  52.8% 중립  19.6% | 흐리 / 사진 / 연하
  T40 [4,102건] 부정  27.4% 긍정  49.2% 중립  23.5% | 팔길이 / 소매 / 몸통
  T34 [2,544건] 부정  21.1% 긍정  66.7% 중립  12.2% | 사람 / 이유 / 인기
  T39 [9,303건] 부정  20.1% 긍정  60.9% 중립  19.0% | 허리 / 스트링 / 허벅지
  T61 [9,607건] 부정  18.9% 긍정  48.7% 중립  32.4% | 작게 / 치수 / 사이즈 작게
  T21 [1,596건] 부정  18.3% 긍정  63.8% 중립  17.9% | 린넨 / 린넨 셔츠 / 린넨 소재


## 6. 평점 vs 감성 교차 검증

In [22]:
# 평점 컬럼 확인
rating_col = '평점' if '평점' in df.columns else None

if rating_col:
    print('=== 평점 vs 감성 교차표 (비율 %) ===')
    cross = pd.crosstab(
        df[rating_col].fillna(-1).astype(int),
        df['sentiment'],
        normalize='index'
    ) * 100
    print(cross.round(1).to_string())
else:
    print('평점 컬럼 없음 — 교차 검증 생략')

=== 평점 vs 감성 교차표 (비율 %) ===
sentiment  negative  neutral  positive
평점                                    
-1              5.5     17.6      76.9
 1             90.7      3.9       5.4
 2             86.0      7.9       6.1
 3             55.4     14.7      29.9
 4             23.5     15.7      60.9
 5              5.9      7.9      86.2


## 7. 감성별 샘플 리뷰

In [23]:
text_col = '리뷰내용_clean' if '리뷰내용_clean' in df.columns else 'sentence'

for label in df['sentiment'].unique():
    sub = df[df['sentiment'] == label]
    print(f'\n[{label}] 총 {len(sub):,}건 — 샘플 3개')
    for _, r in sub.sample(min(3, len(sub)), random_state=42).iterrows():
        rating = f"{r['평점']:.0f}점" if '평점' in df.columns and pd.notna(r['평점']) else ''
        print(f'  ({rating}) {str(r[text_col])[:80]}')


[positive] 총 502,332건 — 샘플 3개
  (5점) 남자친구 사줬는데 딱 맞아요 연청도 허나 살걸그럈어요
  (4점) 색도괜찮고 핏도 좋아요 후기에 크게 나온다고해서 평균사이즈로 구매했는데 다음에살땐 한단계 업해서 입으려구요
  (5점) 만족스러워요 정사이즈로가면 단품으로입어도좋겠어요 전투용으로 입기 딱좋을듯

[negative] 총 63,631건 — 샘플 3개
  (2점) 처음 받을때 검은 실 뭉친거 같은게 안쪽에 좀 많이 묻어있어서 그러려니 하고 빨았는데도 계속 묻어나오네요.
  (5점) 진짜 이뻐보여서 사긴했는데 그만큼 길거리에서 많은 사람들이 입고 있네요 아쉬워요 ㅠ 그래도 이쁩니다 !
  (4점) 평소에 차콜 색 옷 갖고시펏는데 프린팅더 이쁘구ㅠㅠ 근데 세탁하면 물 좀 빠쟈요ㅠ

[neutral] 총 57,417건 — 샘플 3개
  (4점) 전체적으로 큰감이 있어요 m사이즈 시켜도 될것 같아요 기장을 생각하신다면 l사이즈 가셔야합니다
  (5점) 따듯하고 많이 오버 느낌이에요 나쁘지는 않네요
  (5점) 어떤 스타일이던 잘 어울리고, 까칠까칠하지 않아서 입기에 부담 없었습니다.


---

In [24]:
# ── 토픽 배정 리뷰 감성별 샘플 ──────────────────────────────
print('=' * 60)
print('[토픽 배정] 전체 감성 분포')
print('=' * 60)
dist = topic_df['sentiment'].value_counts()
for label, cnt in dist.items():
    print(f'  {label:10s}: {cnt:>7,}건 ({cnt/len(topic_df)*100:.1f}%)')

for label in ['positive', 'negative', 'neutral']:
    sub = topic_df[topic_df['sentiment'] == label]
    print(f'\n[토픽 배정 / {label}] 총 {len(sub):,}건 — 샘플 10개')
    for _, r in sub.sample(min(10, len(sub)), random_state=42).iterrows():
        prob = r['p_pos'] if label == 'positive' else r['p_neg'] if label == 'negative' else max(r['p_pos'], r['p_neg'])
        print(f"  p={prob:.2f} | {r['평점']:.0f}점 | T{int(r['topic_id']):2d} | {r['리뷰내용_clean'][:75]}")

print('\n[토픽 배정] 5점인데 negative')
mis1 = topic_df[(topic_df['평점'] == 5) & (topic_df['sentiment'] == 'negative')]
print(f'총 {len(mis1):,}건 — 샘플 10개')
for _, r in mis1.sample(min(10, len(mis1)), random_state=42).iterrows():
    print(f"  p_neg={r['p_neg']:.2f} | T{int(r['topic_id']):2d} | {r['리뷰내용_clean'][:75]}")

print('\n[토픽 배정] 1~2점인데 positive')
mis2 = topic_df[(topic_df['평점'] <= 2) & (topic_df['sentiment'] == 'positive')]
print(f'총 {len(mis2):,}건 — 샘플 10개')
for _, r in mis2.sample(min(10, len(mis2)), random_state=42).iterrows():
    print(f"  p_pos={r['p_pos']:.2f} | T{int(r['topic_id']):2d} | {r['리뷰내용_clean'][:75]}")

[토픽 배정] 전체 감성 분포
  positive  : 337,181건 (82.2%)
  negative  :  37,563건 (9.2%)
  neutral   :  35,242건 (8.6%)

[토픽 배정 / positive] 총 337,181건 — 샘플 10개
  p=0.91 | 5점 | T 6 | 이런 포인트 있는 바지 좋아하진 않지만 색감이랑 핏이 너무 맘에들어서 사버렸습니다 길고 크긴한데 감당가능
  p=0.97 | 5점 | T51 | 가성비 좋아요 핏도 괜찮고 너무 이쁘네요 마음에 들어요
  p=0.97 | 5점 | T 2 | 진짜 잘입고 다녀요 완전 편해요 배송도 빠르게 왔어여
  p=0.94 | 5점 | T41 | 가격대비 퀄리티 우수하고 디자인도 좋아서 데일리로 입기에 적당하네요
  p=0.96 | 5점 | T19 | 사진이 둘다 비슷한색이라 비슷하게나왓는데 실제로보면 아이보리 색감 엄청좋아요! 그리고 옷재질도 되게좋습니다! 169cm에62kg남자입니
  p=0.97 | 5점 | T13 | 세일해서 샀는데 너무 예뻐요 기모가 들어가서 따뜻하고 지금 입기엔 좀 덥지만 가을부터 또 입을려구요
  p=0.97 | 4점 | T51 | 이쁩니다 색도 이쁘고 네온 하나 장만하려고 했었는데 잘 산 듯 합니다!
  p=0.97 | 5점 | T64 | 일단 178/68이고 소재가 너무 부들부들하니 좋아요 사이즈도 딱 찾던 사이즈구요 완전 만족이요
  p=0.94 | 5점 | T41 | 데일리하게 잘 나온 제품이라 자주 입을 수 있어요
  p=0.98 | 5점 | T62 | 예쁘게 잘 입고 다니고 있습니다. 조금 크지만 예뻐요

[토픽 배정 / negative] 총 37,563건 — 샘플 10개
  p=0.61 | 5점 | T 0 | 남자 170cm 62kg 동생 사줬는데 오버하게 잘 맞았어요 기대보다 질이 좋고 기모도 톡톡하니 잘 들어있어서 제 것도 사고 싶었는데 
  p=0.91 | 5점 | T19 | 다크그레이가 다크가 아니고 그냥 그레이 같아서 아쉽네요 

In [25]:
# ── 아웃라이어 감성별 샘플 ──────────────────────────────────
print('=' * 60)
print('[아웃라이어] 전체 감성 분포')
print('=' * 60)
dist = outlier_df['sentiment'].value_counts()
for label, cnt in dist.items():
    print(f'  {label:10s}: {cnt:>7,}건 ({cnt/len(outlier_df)*100:.1f}%)')

for label in ['positive', 'negative', 'neutral']:
    sub = outlier_df[outlier_df['sentiment'] == label]
    print(f'\n[아웃라이어 / {label}] 총 {len(sub):,}건 — 샘플 10개')
    for _, r in sub.sample(min(10, len(sub)), random_state=42).iterrows():
        prob = r['p_pos'] if label == 'positive' else r['p_neg'] if label == 'negative' else max(r['p_pos'], r['p_neg'])
        print(f"  p={prob:.2f} | {r['평점']:.0f}점 | {r['리뷰내용_clean'][:75]}")

print('\n[아웃라이어] 5점인데 negative')
mis3 = outlier_df[(outlier_df['평점'] == 5) & (outlier_df['sentiment'] == 'negative')]
print(f'총 {len(mis3):,}건 — 샘플 10개')
for _, r in mis3.sample(min(10, len(mis3)), random_state=42).iterrows():
    print(f"  p_neg={r['p_neg']:.2f} | {r['리뷰내용_clean'][:75]}")

print('\n[아웃라이어] 1~2점인데 positive')
mis4 = outlier_df[(outlier_df['평점'] <= 2) & (outlier_df['sentiment'] == 'positive')]
print(f'총 {len(mis4):,}건 — 샘플 10개')
for _, r in mis4.sample(min(10, len(mis4)), random_state=42).iterrows():
    print(f"  p_pos={r['p_pos']:.2f} | {r['리뷰내용_clean'][:75]}")

[아웃라이어] 전체 감성 분포
  positive  : 165,151건 (77.4%)
  negative  :  26,068건 (12.2%)
  neutral   :  22,175건 (10.4%)

[아웃라이어 / positive] 총 165,151건 — 샘플 10개
  p=0.92 | 3점 | 제 키가 작아소 생각보단 컷지만 편하게 입고 다닐때 좋아요
  p=0.98 | 5점 | 너무너무 이뻐요!!! 사이즈도 딱 제가 원하던 사이즈예용!!! 무난하게 입기 딱 좋은거 같아용!!
  p=0.88 | 5점 | 진짜 사지마셈 나만 입을거임 핏이 너무 마음에 듦 핏 사기템 할머니가 골덴 따숩다고 좋아하심
  p=0.98 | 5점 | 하단에 조일수 있는게 있어서 넘 좋고 팔길이가 잘맞아서 좋아요
  p=0.98 | 5점 | 158에m 딱좋아요 길까봐 걱정했는데 엉덩이 딱덮고 이뻐요 질도 부들보들하고 따땃합니당 자주입을거같아용
  p=0.73 | 5점 | 옷 사이즈는 넉넉하게 입을수있을 정도고 마무리도 잘되어있어요
  p=0.96 | 5점 | 이너 아무거나 입어도 예뻤습니당,,, 정말 최고의 남방
  p=0.98 | 5점 | 허리가 밴딩이라서 편하고 여름에 입으니까 시원하고 이뻐요.1+1으로 가격도 너무 착해요
  p=0.93 | 5점 | 데일리로 이너로 입기 딱좋아요! 니트소재인줄알았는데 약간 면느낌이에요
  p=0.94 | 5점 | 정말잘 입을것 같아요 기장이 조금 기네요 다음에도 꼭 사서 입고싶네요

[아웃라이어 / negative] 총 26,068건 — 샘플 10개
  p=0.98 | 3점 | 구매량이 많아서 가을에 버티기용으로 샀었는데 생각보다 아쉬웠습니다 가격이 싸긴하지만 다시 마원 구매해야할때는 비싼거를 사거나 다른 브랜
  p=0.56 | 5점 | 요즘 셔츠 코디가 이뻐 사서 입어봤는데 이쁘게 핏이 떨어지는 거 같아요 생각보다 두꺼워서 날 시원할 때 입으면 좋을 것 같습니다~~
  p=0.84 | 4점 | 제목그대로 기장이 조금 길어요 두께감이나 원단질감은